In [12]:
import sqlite3
import pandas as pd

In [13]:
conn = sqlite3.connect('../dataset/nfts.sqlite/nfts.sqlite')

In [14]:
print("Loading transfers data...")

# transfers = pd.read_sql_query("""
#     SELECT 
#         transaction_hash,
#         timestamp,
#         nft_address,
#         token_id,
#         from_address,
#         to_address,
#         transaction_value
#     FROM transfers
#     LIMIT 500000
# """, conn)

transfers = pd.read_sql_query("""
    SELECT 
        transaction_hash,
        timestamp,
        nft_address,
        token_id,
        from_address,
        to_address,
        transaction_value
    FROM transfers
    WHERE timestamp >= 1627776000  -- August 1, 2021
    AND timestamp < 1633046400     -- October 1, 2021
    LIMIT 1000000
""", conn)

print(f"Loaded: {len(transfers):,} rows")
print(f"Columns: {transfers.columns.tolist()}")
print(f"\nSample:\n{transfers.head(3)}")

Loading transfers data...
Loaded: 1,000,000 rows
Columns: ['transaction_hash', 'timestamp', 'nft_address', 'token_id', 'from_address', 'to_address', 'transaction_value']

Sample:
                                    transaction_hash   timestamp  \
0  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...  1627776481   
1  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...  1627776481   
2  0x0c73daad5e6946e6b2f1374846d6ba3025d6e6a2569f...  1627776481   

                                  nft_address  \
0  0x629A673A8242c2AC4B7B8C5D8735fbeac21A6205   
1  0x629A673A8242c2AC4B7B8C5D8735fbeac21A6205   
2  0x629A673A8242c2AC4B7B8C5D8735fbeac21A6205   

                                            token_id  \
0  6682934142305278177002530107138320246486863356...   
1  1198507443711741845273832738859307575893198744...   
2  1541220344281436903042142177888467401468454835...   

                                 from_address  \
0  0x25f1d709b329C7349b8209851E90eFa3a7f60178   
1  0x0737E7162C88E9FBB963334

In [15]:
print("\n--- Preprocessing ---")

# 2.1 Convert Unix timestamp to datetime
transfers['timestamp'] = pd.to_datetime(
    transfers['timestamp'], unit='s'
)
print(f"Date range: {transfers['timestamp'].min()} "
      f"to {transfers['timestamp'].max()}")

# 2.2 Keep only sales (transaction_value > 0)
# Aligned with Liu et al. (2023) who only analyze paid transactions
sales = transfers[transfers['transaction_value'] > 0].copy()
print(f"Sales only (value > 0): {len(sales):,} "
      f"({len(sales)/len(transfers):.1%} of total)")

# 2.3 Remove burn/null addresses
# These are NFT destructions, not ownership transfers
BURN_ADDRESSES = {
    '0x0000000000000000000000000000000000000000',  # null address
    '0x000000000000000000000000000000000000dead',  # dead address
}

burn_lower = {a.lower() for a in BURN_ADDRESSES}

sales_clean = sales[
    ~sales['from_address'].str.lower().isin(burn_lower) &
    ~sales['to_address'].str.lower().isin(burn_lower)
].copy().reset_index(drop=True)

print(f"After removing burn addresses: {len(sales_clean):,}")
print(f"Removed: {len(sales) - len(sales_clean):,} rows")


--- Preprocessing ---
Date range: 2021-08-01 00:00:17 to 2021-08-24 00:15:28
Sales only (value > 0): 690,986 (69.1% of total)
After removing burn addresses: 685,892
Removed: 5,094 rows


In [16]:
# Initialize label column
sales_clean['is_wash_trading'] = 0
wash_hashes = set()

# Sort by timestamp once globally
sales_clean = sales_clean.sort_values('timestamp').reset_index(drop=True)

# Group by NFT token (each unique NFT)
grouped = sales_clean.groupby(['nft_address', 'token_id'])
total_groups = len(grouped)
print(f"\nTotal unique NFT tokens: {total_groups:,}")

# Constants
MAX_DAYS_BUYBACK  = 30          # Rule 1: Liu et al. (2023)
MAX_HOURS_CYCLE   = 24          # Rule 2: Von Wachter et al. (2022)
MAX_SECS_BUYBACK  = MAX_DAYS_BUYBACK * 24 * 3600
MAX_SECS_CYCLE    = MAX_HOURS_CYCLE * 3600

print(f"\nApplying labeling rules:")
print(f"  Rule 1 (Liu et al. 2023)        : Seller buyback < {MAX_DAYS_BUYBACK} days")
print(f"  Rule 2 (Von Wachter et al. 2022): Direct cycle A->B->A < {MAX_HOURS_CYCLE} hours")
print(f"\nProcessing...")

rule1_count = 0
rule2_count = 0
processed   = 0

for (nft_addr, token_id), group in grouped:

    # Skip tokens with only 1 transaction (no cycle possible)
    if len(group) < 2:
        processed += 1
        continue

    group = group.sort_values('timestamp').reset_index(drop=True)
    n = len(group)

    for i in range(n):
        from_i = group.loc[i, 'from_address'].lower()
        to_i   = group.loc[i, 'to_address'].lower()
        hash_i = group.loc[i, 'transaction_hash']
        ts_i   = group.loc[i, 'timestamp']

        for j in range(i + 1, n):
            ts_j      = group.loc[j, 'timestamp']
            diff_secs = (ts_j - ts_i).total_seconds()

            # Early stop: beyond max buyback window
            if diff_secs > MAX_SECS_BUYBACK:
                break

            from_j = group.loc[j, 'from_address'].lower()
            to_j   = group.loc[j, 'to_address'].lower()
            hash_j = group.loc[j, 'transaction_hash']

            # --------------------------------------------------
            # RULE 1: Seller Buyback (Liu et al., 2023)
            # Wallet that sold NFT buys it back within 30 days
            # Pattern: A sells to anyone, then A buys back
            # --------------------------------------------------
            if from_i == to_j:
                if hash_i not in wash_hashes or hash_j not in wash_hashes:
                    wash_hashes.add(hash_i)
                    wash_hashes.add(hash_j)
                    rule1_count += 1

            # --------------------------------------------------
            # RULE 2: Direct Cycle (Von Wachter et al., 2022)
            # NFT goes A->B then B->A within 24 hours
            # Pattern: exact round-trip in short time
            # --------------------------------------------------
            if (from_i == to_j and
                to_i   == from_j and
                diff_secs <= MAX_SECS_CYCLE):
                if hash_i not in wash_hashes or hash_j not in wash_hashes:
                    wash_hashes.add(hash_i)
                    wash_hashes.add(hash_j)
                    rule2_count += 1

    processed += 1
    if processed % 20000 == 0:
        print(f"  Progress: {processed:,} / {total_groups:,} tokens "
              f"({processed/total_groups:.1%})")

# Apply labels to dataframe
sales_clean.loc[
    sales_clean['transaction_hash'].isin(wash_hashes),
    'is_wash_trading'
] = 1


Total unique NFT tokens: 553,519

Applying labeling rules:
  Rule 1 (Liu et al. 2023)        : Seller buyback < 30 days
  Rule 2 (Von Wachter et al. 2022): Direct cycle A->B->A < 24 hours

Processing...
  Progress: 140,000 / 553,519 tokens (25.3%)
  Progress: 240,000 / 553,519 tokens (43.4%)
  Progress: 280,000 / 553,519 tokens (50.6%)
  Progress: 520,000 / 553,519 tokens (93.9%)


In [20]:
# ============================================================
# DEBUG: Why Rule 2 (Direct Cycle A->B->A) caught nothing?
# ============================================================

print("Investigating Rule 2 (Direct Cycle A->B->A < 24 hours)...")

# Look at the sample wash trading transactions
# Row 11759 and 13788 look like a cycle!
# 0xcB55... -> 0x8864... (16:29)
# 0x8864... -> 0xcB55... (18:00)
# That's only ~1.5 hours apart — should be caught!

# Let's manually check this specific NFT
example = sales_clean[
    (sales_clean['from_address'].str.lower() == 
     '0xcb55aafaabf60d16f8c7f31c0880bf508bf2a858') |
    (sales_clean['to_address'].str.lower() == 
     '0xcb55aafaabf60d16f8c7f31c0880bf508bf2a858')
].sort_values('timestamp')

print(f"\nTransactions involving 0xcB55...:")
print(example[['timestamp', 'nft_address', 'token_id',
               'from_address', 'to_address',
               'transaction_value', 'is_wash_trading']].to_string())

# Check: are they the same NFT token?
print(f"\nUnique token_ids: {example['token_id'].unique()}")
print(f"Unique nft_address: {example['nft_address'].unique()}")

Investigating Rule 2 (Direct Cycle A->B->A < 24 hours)...

Transactions involving 0xcB55...:
                 timestamp                                 nft_address  token_id                                from_address                                  to_address  transaction_value  is_wash_trading
11759  2021-08-01 16:29:51  0xa7d8d9ef8D8Ce8992Df33D8b8CF4Aebabd5bD270  34000944  0xcB55AAfAaBf60d16F8c7F31c0880bF508Bf2a858  0x886478D3cf9581B624CB35b5446693Fc8A58B787       1.378000e+17                1
13788  2021-08-01 18:00:13  0xa7d8d9ef8D8Ce8992Df33D8b8CF4Aebabd5bD270  34000944  0x886478D3cf9581B624CB35b5446693Fc8A58B787  0xcB55AAfAaBf60d16F8c7F31c0880bF508Bf2a858       1.378000e+17                1
149127 2021-08-07 15:44:51  0x50f5474724e0Ee42D9a4e711ccFB275809Fd6d4a    164487  0xcB55AAfAaBf60d16F8c7F31c0880bF508Bf2a858  0x944fdeA9d4956ce673C7545862cefCcad6Ee1B04       3.590000e+17                0
273488 2021-08-11 05:13:04  0xf3E6DbBE461C6fa492CeA7Cb1f5C5eA660EB1B47      8845  0xcB5

In [17]:
total     = len(sales_clean)
wt_count  = sales_clean['is_wash_trading'].sum()
norm_count = total - wt_count
ratio     = wt_count / total

print(f"\n{'='*50}")
print(f"LABELING RESULTS")
print(f"{'='*50}")
print(f"Total sales transactions : {total:,}")
print(f"Wash trading (label=1)   : {wt_count:,} ({ratio:.3%})")
print(f"Normal (label=0)         : {norm_count:,} ({1-ratio:.3%})")
print(f"\nBreakdown by rule:")
print(f"  Rule 1 (seller buyback): {rule1_count:,} pairs flagged")
print(f"  Rule 2 (direct cycle)  : {rule2_count:,} pairs flagged")

# Sanity check: no burn addresses in wash trading labels
wt_df = sales_clean[sales_clean['is_wash_trading'] == 1]
burn_in_wt = wt_df[
    wt_df['to_address'].str.lower().isin(burn_lower)
]
print(f"\nSanity check:")
print(f"  Burn addresses in wash trading labels: {len(burn_in_wt)}")
print(f"  (Expected: 0)")

# Sample wash trading transactions
print(f"\nSample wash trading transactions:")
print(wt_df[['timestamp', 'from_address',
             'to_address', 'transaction_value',
             'is_wash_trading']].head(5).to_string())


LABELING RESULTS
Total sales transactions : 685,892
Wash trading (label=1)   : 2,991 (0.436%)
Normal (label=0)         : 682,901 (99.564%)

Breakdown by rule:
  Rule 1 (seller buyback): 380 pairs flagged
  Rule 2 (direct cycle)  : 0 pairs flagged

Sanity check:
  Burn addresses in wash trading labels: 0
  (Expected: 0)

Sample wash trading transactions:
                timestamp                                from_address                                  to_address  transaction_value  is_wash_trading
1752  2021-08-01 02:15:24  0x3155f38f8F24E15562F38533eaC9a123c21334Ef  0x42dE10A720c59eD8dcC6E55d5E61e03B5AD70905       3.100000e+18                1
4329  2021-08-01 06:01:27  0xa5F7A90bAD45094aa676f81D580D8B93284E8506  0x6f6dad241276eeddeB5500B5E896c57266f9fFA7       3.300000e+17                1
7151  2021-08-01 11:23:57  0xF0B6339404cE990A9b9A7B940989b111Fc4E268c  0x0356a9f78A521f4456F255d848624d90a5B91fec       4.000000e+17                1
11759 2021-08-01 16:29:51  0xcB55AAfAaBf60d

In [18]:
# Check total data and date range
print("Checking full dataset...")

# Total rows
total_rows = pd.read_sql_query(
    "SELECT COUNT(*) as count FROM transfers", conn
).iloc[0]['count']
print(f"Total rows in transfers: {total_rows:,}")

# Date range
date_info = pd.read_sql_query("""
    SELECT 
        MIN(timestamp) as min_ts,
        MAX(timestamp) as max_ts,
        COUNT(*) as total
    FROM transfers
""", conn)

min_ts = pd.to_datetime(date_info['min_ts'].iloc[0], unit='s')
max_ts = pd.to_datetime(date_info['max_ts'].iloc[0], unit='s')

print(f"Date range: {min_ts} to {max_ts}")
print(f"Duration: {(max_ts - min_ts).days} days "
      f"({(max_ts - min_ts).days / 30:.1f} months)")

# Distribution per month
monthly = pd.read_sql_query("""
    SELECT 
        strftime('%Y-%m', datetime(timestamp, 'unixepoch')) as month,
        COUNT(*) as count
    FROM transfers
    GROUP BY month
    ORDER BY month
""", conn)

print(f"\nTransactions per month:")
print(monthly.to_string(index=False))

Checking full dataset...
Total rows in transfers: 4,514,729
Date range: 2021-04-01 00:00:14 to 2021-09-25 16:15:40
Duration: 177 days (5.9 months)

Transactions per month:
  month   count
2021-04  222822
2021-05  222012
2021-06  376631
2021-07  597449
2021-08 1560478
2021-09 1535337


In [19]:
# Option: Load August + September (peak NFT boom)
# These months have most activity = most wash trading
print("Loading peak period data (August - September 2021)...")

# August 1 = Unix 1627776000
# October 1 = Unix 1633046400

transfers_peak = pd.read_sql_query("""
    SELECT 
        transaction_hash,
        timestamp,
        nft_address,
        token_id,
        from_address,
        to_address,
        transaction_value
    FROM transfers
    WHERE timestamp >= 1627776000  -- August 1, 2021
    AND timestamp < 1633046400     -- October 1, 2021
    LIMIT 1000000
""", conn)

print(f"Loaded: {len(transfers_peak):,} rows")

# Check distribution
transfers_peak['timestamp_dt'] = pd.to_datetime(
    transfers_peak['timestamp'], unit='s'
)
monthly = transfers_peak.groupby(
    transfers_peak['timestamp_dt'].dt.to_period('M')
).size()
print(f"\nDistribution:")
print(monthly)

Loading peak period data (August - September 2021)...
Loaded: 1,000,000 rows

Distribution:
timestamp_dt
2021-08    1000000
Freq: M, dtype: int64
